## Lab 02 : Sequence-to-sequence transformers -- exercise 

The primary objective is to build a sequence-to-sequence (Seq2Seq) Transformer architecture that learns to "translate" an unordered sequence of integers into a sorted output sequence.

Here, the input sequence is a random sequence of integers, e.g. [12,  8, 10,  5,  7, 13,  4,  1,  6,  2].

The output sequence is the sorted version of the input sequence, e.g. [ 1,  2,  4,  5,  6,  7,  8, 10, 12, 13].

Implement:

Architecture Components: It implements the original Transformer design from "Attention Is All You Need," including Positional Encodings, Multi-Head Self-Attention for the encoder, and Masked Multi-Head Self/Cross-Attention for the decoder.

Training Pipeline: The notebook features a training loop using mini-batches, an Adam optimizer, and a learning rate scheduler to minimize Cross-Entropy Loss over 100 epochs.

Evaluation: It demonstrates auto-regressive generation, where the model predicts the sorted sequence one token at a time based on previously generated tokens, achieving 100% accuracy on unseen data.

In [9]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5242_2026_codes/labs_lecture06/lab02_seq2seq/'
    print(path_to_file)
    # change current path to the folder containing "file_name"
    os.chdir(path_to_file)
    !pwd
    

In [10]:
# import torch
# import torch.nn.functional as F
# import torch.nn as nn
# import torch.optim as optim
# import math
# import time

# Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt
import logging
import math
logging.getLogger().setLevel(logging.CRITICAL) # remove warnings
import os, datetime

# PyTorch version and GPU
print(torch.__version__)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))
  device= torch.device("cuda:0") # use GPU
else:
  device= torch.device("cpu")
print(device)


2.1.2.post304
NVIDIA GeForce RTX 5080
cuda:0


## Dataset

In [11]:
# Minibatch
batch_size = bs = 4
seq_length = 15
seq_enc_in = torch.stack([torch.randperm(seq_length)[:10] for _ in range(batch_size)]).to(device)
print('seq_enc_in:', seq_enc_in.size(), seq_enc_in)
seq_target = seq_enc_in.sort(dim=1)[0]
print('seq_target:', seq_target.size(), seq_target)
BOS = seq_length   # start token is BOS=seq_length=15
start_tokens = torch.empty(batch_size, 1, dtype=torch.long).fill_(BOS).to(device)
seq_dec_in = torch.cat((start_tokens, seq_target), dim=1)
EOS = seq_length+1 # start token is BOS=seq_length+1=16
end_tokens = torch.empty(batch_size, 1, dtype=torch.long).fill_(EOS).to(device)
seq_dec_out = torch.cat((seq_target, end_tokens), dim=1)
print('seq_enc_in:', seq_enc_in.size(), seq_enc_in)
print('seq_dec_in:', seq_dec_in.size(), seq_dec_in)
print('seq_dec_out:', seq_dec_out.size(), seq_dec_out)


seq_enc_in: torch.Size([4, 10]) tensor([[11,  0,  7,  6,  9,  5,  3, 10, 14, 13],
        [ 2,  5,  9,  6, 11,  1,  8, 12,  4, 10],
        [ 4,  0, 13, 10,  3,  7,  8, 12, 14,  6],
        [ 4,  5,  9, 12,  1, 11,  2, 13,  8,  0]], device='cuda:0')
seq_target: torch.Size([4, 10]) tensor([[ 0,  3,  5,  6,  7,  9, 10, 11, 13, 14],
        [ 1,  2,  4,  5,  6,  8,  9, 10, 11, 12],
        [ 0,  3,  4,  6,  7,  8, 10, 12, 13, 14],
        [ 0,  1,  2,  4,  5,  8,  9, 11, 12, 13]], device='cuda:0')
seq_enc_in: torch.Size([4, 10]) tensor([[11,  0,  7,  6,  9,  5,  3, 10, 14, 13],
        [ 2,  5,  9,  6, 11,  1,  8, 12,  4, 10],
        [ 4,  0, 13, 10,  3,  7,  8, 12, 14,  6],
        [ 4,  5,  9, 12,  1, 11,  2, 13,  8,  0]], device='cuda:0')
seq_dec_in: torch.Size([4, 11]) tensor([[15,  0,  3,  5,  6,  7,  9, 10, 11, 13, 14],
        [15,  1,  2,  4,  5,  6,  8,  9, 10, 11, 12],
        [15,  0,  3,  4,  6,  7,  8, 10, 12, 13, 14],
        [15,  0,  1,  2,  4,  5,  8,  9, 11, 12, 13]], d

## Design the original Seq-To-Seq Transformer network

Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin, Attention Is All You Need, https://arxiv.org/pdf/1706.03762

In [12]:
vocab_size = seq_length + 2 # +2 for the begin/start token (BOS) and the end token (EOS)


###############################
# Positional Encoding
###############################
def generate_positional_encoding(seq_length, dim):
    assert dim == 2* (dim//2) # check if dim is divisible by 2
    pe = torch.zeros(seq_length, dim)
    position = torch.arange(0, seq_length, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / dim))
    pe[:,0::2] = torch.sin(position * div_term)
    pe[:,1::2] = torch.cos(position * div_term)
    return pe        
    

###############################
# Encoder Transformer
###############################
class AttentionHead_encoder(nn.Module):
    def __init__(self, d, d_head):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); batch_len = H.size(1)
        # self-attention encoder
        # Compute a single attention head H = Softmax( QK^T / d^0.5 ) V
        Q = self.query(H) # size=[batch_size, batch_length, d] 
        K = self.key(H) # size=[batch_size, batch_length, d]
        V = self.value(H) # size=[batch_size, batch_length, d]

        # Compute the attension score matrix = Q x K^T / sqrt(d), where "x" is the matrix-matrix multiplication
        # Hint: "x" can be implemented with PyTorch "@" and ".transpose(2,1)" for transpose tensor operator
        # COMPLETE HERE
        attention_score = (Q @ K.transpose(1,2))/math.sqrt(Q.size(2))   # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        # COMPLETE HERE
        
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len)
        H_HA = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        return H_HA # return prediction scores for next token
        
class MultipleAttentionHead_encoder(nn.Module):
    def __init__(self, d, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d // num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ AttentionHead_encoder(d, d_head) for _ in range(num_heads) ])
        self.WO = nn.Linear(d, d) # combination layer
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); seq_length = H.size(1)
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, seq_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, seq_length, d]            
        H = self.WO(H_heads) # size=[batch_size, seq_length, d]
        return H

class TransformerBlock_encoder(nn.Module):
    def __init__(self, d, num_heads):
        super().__init__()
        self.LN_MHA = nn.LayerNorm(d)
        self.LN_MLP = nn.LayerNorm(d)
        self.MHA = MultipleAttentionHead_encoder(d, num_heads)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))   
    def forward(self, H): # size=[batch_size, seq_length, d]

        # Apply layer normalization, multi-head attention, and residual connection to hidden features
        # COMPLETE HERE
        H = H + self.MHA(self.LN_MHA(H))   # Self-attention encoder, ize=[batch_size, seq_length, d], [20, 30, 128]
        # COMPLETE HERE

        # Apply layer normalization, multi-layer perceptron, and residual connection to hidden features
        # COMPLETE HERE
        H = H + self.MLP(self.LN_MLP(H))    # MLP, size=[batch_size, seq_length, d]
        # COMPLETE HERE
        
        return H # size=[batch_size, seq_length, d]
    
class Transformer_encoder(nn.Module):
    def __init__(self, d, num_heads, num_blocks, seq_length):
        super().__init__()
        self.TR_Blocks = nn.ModuleList([ TransformerBlock_encoder(d, num_heads) for _ in range(num_blocks) ]) 
    def forward(self, batch_seq, pos_enc):
        H = batch_seq # size=[batch_size, seq_length, d]
        batch_len = H.size(1)
        pos_enc = pos_enc.unsqueeze(dim=0) # positional encoding, size=[1, seq_length, d]

        # Add the positional encoding to the embedded tokens to tell the network their positions in the sequence
        # Hint : slice pos_enc up to "batch_len" size
        # COMPLETE HERE
        H = H + pos_enc[:, :batch_len]     # size=[batch_size, seq_length, d]
        # COMPLETE HERE
        
        for TR_Block in self.TR_Blocks: # Apply transformer blocks 
            H = TR_Block(H)
        return H # return prediction scores for next token, size=[batch_length, batch_size, d]

    
###############################
# Decoder Transformer
###############################
class SelfAttention_AttentionHead_decoder(nn.Module):
    def __init__(self, d, d_head):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); batch_len = H.size(1)
        # Masked self-attention decoder
        # Compute a single attention head H = Softmax( QK^T / d^0.5 ) V
        Q = self.query(H) # size=[batch_size, batch_length, d] 
        K = self.key(H) # size=[batch_size, batch_length, d]
        V = self.value(H) # size=[batch_size, batch_length, d]
        attention_score = Q @ K.transpose(2,1) * Q.size(2)**-0.5 # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)

        # Prepare a mask to hide future tokens, but only keep previous tokens, for example for L=4, it is equal to
        # mask = [ [1, 0, 0, 0],
        #          [1, 1, 0, 0],
        #          [1, 1, 1, 0],
        #          [1, 1, 1, 1] ]
        # Hint: You may use " torch.tril(X)" that returns the lower triangular part of matrix X of size (L,L)
        #       and "torch.ones(L,L)" returns a tensor of size (L,L) filled with the scalar value 1.0,
        #       as well as ".long()" and ".to(device)" if necessary
        # COMPLETE HERE
        mask = torch.tril(torch.ones(batch_len,batch_len)).long().to(device)    # mask to use previous tokens only : { token(<=t) }, size=[batch_len,batch_len]
        # COMPLETE HERE
        
        attention_score = attention_score.masked_fill(mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len)
        H_HA = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        return H_HA # return prediction scores for next token
    
class SelfAttention_MultipleAttentionHead_decoder(nn.Module):
    def __init__(self, d, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d // num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ SelfAttention_AttentionHead_decoder(d, d_head) for _ in range(num_heads) ])
        self.WO = nn.Linear(d, d) # combination layer
    def forward(self, H): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); seq_length = H.size(1)
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, seq_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, seq_length, d]            
        H = self.WO(H_heads) # size=[batch_size, seq_length, d]
        return H

class CrossAttention_AttentionHead_decoder(nn.Module):
    def __init__(self, d, d_head):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
    def forward(self, H, Henc): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); batch_len = H.size(1)
        # Masked cross-attention
        # Compute a single attention head H = Softmax( QK^T / d^0.5 ) V
        Q = self.query(H) # size=[batch_size, batch_length, d]  
        K = self.key(Henc) # size=[batch_size, batch_length, d]
        V = self.value(Henc) # size=[batch_size, batch_length, d]
        attention_score = Q @ K.transpose(2,1) * Q.size(2)**-0.5 # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len)
        H_HA = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        return H_HA # return prediction scores for next token
        
class CrossAttention_MultipleAttentionHead_decoder(nn.Module):
    def __init__(self, d, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d // num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ CrossAttention_AttentionHead_decoder(d, d_head) for _ in range(num_heads) ])
        self.WO = nn.Linear(d, d) # combination layer
    def forward(self, H, Henc): # size(H)=[batch_size, seq_length, d]
        batch_size = H.size(0); seq_length = H.size(1)
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H, Henc)) # size=[batch_size, seq_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, seq_length, d]            
        H = self.WO(H_heads) # size=[batch_size, seq_length, d]
        return H
    
class TransformerBlock_decoder(nn.Module):
    def __init__(self, d, num_heads):
        super().__init__()
        self.LN_MHA_H = nn.LayerNorm(d)
        self.LN_MHA_Henc = nn.LayerNorm(d)
        self.LN_MLP = nn.LayerNorm(d)
        self.SA_MHA = SelfAttention_MultipleAttentionHead_decoder(d, num_heads)
        self.CA_MHA = CrossAttention_MultipleAttentionHead_decoder(d, num_heads)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))   
    def forward(self, H, Henc): # size=[batch_size, seq_length, d]
        H = H + self.SA_MHA(self.LN_MHA_H(H)) # Masked self-attention decoder, size=[batch_size, seq_length, d], [20, 30, 128]
        H = H + self.CA_MHA(self.LN_MHA_H(H),self.LN_MHA_Henc(Henc)) # Masked cross-attention decoder, size=[batch_size, seq_length, d], [20, 30, 128]
        H = H + self.MLP(self.LN_MLP(H)) # MLP, size=[batch_size, seq_length, d]
        return H # size=[batch_size, seq_length, d]
           
class Transformer_decoder(nn.Module):
    def __init__(self, d, num_heads, num_blocks, seq_length):
        super().__init__()
        self.TR_Blocks = nn.ModuleList([ TransformerBlock_decoder(d, num_heads) for _ in range(num_blocks) ]) 
    def forward(self, g_seq_out, h_enc_seq, pos_enc):
        H = g_seq_out # size=[batch_size, seq_length+1, d], e.g. [4, 16, 128]
        Henc = h_enc_seq # size=[batch_size, seq_length, d], e.g. [4, 15, 128]
        batch_len = H.size(1)
        pos_enc = pos_enc.unsqueeze(dim=0) # positional encoding, size=[1, seq_length, d]
        H = H + pos_enc[:,:batch_len]      # size=[batch_size, seq_length+1, d], e.g. [4, 16, 128]
        for TR_Block in self.TR_Blocks: # Apply transformer blocks 
            H = TR_Block(H, Henc)
        return H # return prediction scores for next token, size=[batch_length, batch_size, d]


###############################
# Encoder-Decoder Transformer
###############################    
class EncDecTransformer(nn.Module):
    def __init__(self, d, num_heads, num_blocks, seq_length):
        super(EncDecTransformer, self).__init__()
        self.encoder = Transformer_encoder(d, num_heads, num_blocks, seq_length)
        self.decoder = Transformer_decoder(d, num_heads, num_blocks, seq_length)
    def forward(self, g_enc_in , g_dec_in, pos ):
        h_enc_out = self.encoder( g_enc_in , pos ) # size=[batch_size, seq_length, d], [4, 15, 128]
        h_dec_out = self.decoder( g_dec_in, h_enc_out , pos )  # size=[batch_size, seq_length+1, d], e.g. [4, 16, 128]
        return h_dec_out 

###############################
# "Attention Is All You Need" Model
# https://arxiv.org/pdf/1706.03762
############################### 
class attention_net(nn.Module):
    def __init__(self, d, num_heads, num_blocks, seq_length):
        super(attention_net, self).__init__()  
        self.layer1 = nn.Embedding( vocab_size  , hidden_size  )
        self.layer2 = EncDecTransformer(d, num_heads, num_blocks, seq_length)
        self.layer3 = nn.Linear(    hidden_size , vocab_size   )
    def forward(self, seq_enc_in, seq_dec_in, pos_enc ): # seq_enc_in=[bs,seq_length]=[4, 15], seq_dec_in=[bs,seq_length+1,hidden_dim]=[4, 16, 128]
        # Apply the 3 layers for embedding input tokens (here numbers), for applying EncDecTransformer layers, 
        # and projecting back to the dimension of the dictionary of tokens (here the set numbers).
        # COMPLETE HERE
        g_enc_in  = self.layer1(seq_enc_in)    # size=(bs, seq_length, hidden_dim), size=[4, 15, 128]
        g_dec_in  = self.layer1(seq_dec_in)    # size=(bs, seq_length+1, hidden_dim), size=[4, 16, 128]
        h_seq     = self.layer2(g_enc_in, g_dec_in, pos_enc)    # size=(bs, seq_length+1, hidden_dim), size=[4, 16, 128]
        score_seq = self.layer3(h_seq)    # size=(bs, seq_length+1, vocab_size), e.g. [4, 16, 17]
        # COMPLETE HERE
        return score_seq 



# Create a minibatch
batch_size = bs = 4
seq_length = 15
seq_enc_in = torch.stack([torch.randperm(seq_length) for _ in range(batch_size)]).to(device)
seq_target = torch.stack([torch.arange(seq_length) for _ in range(batch_size)]).to(device)
BOS = seq_length   # start token is BOS=seq_length=15
start_tokens = torch.empty(batch_size, 1, dtype=torch.long).fill_(BOS).to(device)
seq_dec_in = torch.cat((start_tokens, seq_target), dim=1)
EOS = seq_length+1 # start token is BOS=seq_length+1=16
end_tokens = torch.empty(batch_size, 1, dtype=torch.long).fill_(EOS).to(device)
seq_dec_out = torch.cat((seq_target, end_tokens), dim=1)
print('seq_enc_in:', seq_enc_in.size(), seq_enc_in)
print('seq_dec_in:', seq_dec_in.size(), seq_dec_in)
print('seq_dec_out:', seq_dec_out.size(), seq_dec_out)

# Test the forward pass 
hidden_size = 128 
num_heads = 16
num_blocks = 2
net = attention_net(hidden_size, num_heads, num_blocks, seq_length).to(device)
pos_enc = generate_positional_encoding(vocab_size, hidden_size).to(device) # size=(vocab_size, hidden_dim), e.g. [4, 16, 17]
print('pos:', pos_enc.size())
scores = net( seq_enc_in, seq_dec_in, pos_enc ) # size=(bs, seq_length+1, vocab_size), e.g. [4, 16, 17]
print('scores:', scores.size())

# Compute the loss
# reshape the scores and labels to huge batch of size bs*seq_length
scores = scores.view(  bs*(seq_length+1) , vocab_size) # size=(seq_length.bs, vocab_size), e.g. [64, 17]
print('scores:', scores.size())
seq_dec_out = seq_dec_out.view(  bs*(seq_length+1) ) # size=(seq_length.bs, vocab_size), e.g. [64]
print('seq_dec_out:', seq_dec_out.size())
criterion = nn.CrossEntropyLoss()
loss = criterion(scores, seq_dec_out)
print('loss:', loss)
loss.backward()
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001)
optimizer.step()



seq_enc_in: torch.Size([4, 15]) tensor([[ 4, 12,  0,  7,  9,  6,  1, 11,  5, 10, 13,  8,  2, 14,  3],
        [14,  4,  8, 10, 13, 12,  7, 11,  0,  3,  6,  1,  9,  2,  5],
        [14,  3,  9,  0,  8,  1,  5, 10,  2, 12,  4,  6, 13, 11,  7],
        [ 4,  9,  1,  0,  3,  2,  8, 10, 13,  5, 14,  7, 11, 12,  6]],
       device='cuda:0')
seq_dec_in: torch.Size([4, 16]) tensor([[15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
        [15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
        [15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
        [15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14]],
       device='cuda:0')
seq_dec_out: torch.Size([4, 16]) tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 16],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11

## Training

In [13]:
# Initialize the net
del net
hidden_size = 128 
num_heads = 16
num_blocks = 2
net = attention_net(hidden_size, num_heads, num_blocks, seq_length).to(device)
pos_enc = generate_positional_encoding(vocab_size, hidden_size).to(device) # size=(vocab_size, hidden_dim), e.g. [4, 16, 17]
# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param
num_param = number_param(net)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Initialize the optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.85, patience=1)
# num_epochs = 5; num_batches = 10 # debug
num_epochs = 100; num_batches = 100

# For batch construction
batch_size = bs = 4
seq_length = 15
BOS = seq_length   # start token is BOS=seq_length=15
start_tokens = torch.empty(batch_size, 1, dtype=torch.long).fill_(BOS).to(device)
EOS = seq_length+1 # start token is BOS=seq_length+1=16
end_tokens = torch.empty(batch_size, 1, dtype=torch.long).fill_(EOS).to(device)

# Train
start = time.time()
for epoch in range(num_epochs):
    
    # set the running quantities to zero at the beginning of the epoch
    running_loss = 0.0  

    for idx_batch in range(num_batches):

        # create a minibatch
        seq_enc_in = torch.stack([torch.randperm(seq_length)[:10] for _ in range(batch_size)]).to(device)
        seq_target = seq_enc_in.sort(dim=1)[0]

        # append BOS and EOS
        seq_dec_in = torch.cat((start_tokens, seq_target), dim=1)
        seq_dec_out = torch.cat((seq_target, end_tokens), dim=1)

        # forward the minibatch through the net     
        scores = net( seq_enc_in, seq_dec_in, pos_enc ) # size=(bs, seq_length+1, vocab_size), e.g. [4, 16, 17]

        # reshape the scores and labels to huge batch of size bs*seq_length
        batch_len = scores.size(1)
        scores = scores.view(  bs*batch_len , vocab_size) # size=(batch_len.bs, vocab_size), e.g. [64, 17]
        seq_dec_out = seq_dec_out.view(  bs*batch_len ) # size=(batch_len.bs, vocab_size), e.g. [64]

        # Compute the average of the losses of the data points in this batch
        loss = criterion(scores, seq_dec_out)

        # backward pass to compute dL/dR, dL/dV and dL/dW
        loss.backward()

        # do one step of stochastic gradient descent: R=R-lr(dL/dR), V=V-lr(dL/dV), ...
        optimizer.step()
        
        # update the running loss  
        running_loss += loss.item()
        
    # compute stats for the full training set
    mean_loss = running_loss / num_batches
    scheduler.step(mean_loss)
    elapsed = time.time() - start
    
    if not epoch%1:
        print(f'epoch: {epoch}, time(min): {elapsed/60:.2f}, lr: {optimizer.param_groups[0]["lr"]:.7f}, mean_loss: {mean_loss:.4f}')

        
# Saving checkpoint
import os, datetime
os.makedirs('checkpoint', exist_ok=True)
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")
checkpoint_file = 'checkpoint/seq2seq_checkpoint_' + time_stamp + '.pkl'
torch.save({
    'epoch': epoch,
    'tot_time_min': elapsed/60,
    'mean_loss': mean_loss,
    'net': net.state_dict(),
    }, checkpoint_file)
print('\ncheckpoint_file:', checkpoint_file)

# epoch: 99, time(min): 13.15, lr: 0.0000003, mean_loss: 0.0060
# checkpoint_file: checkpoint/seq2seq_checkpoint_26-02-18--14-34-00.pkl


num_net_parameters: 928529 / 0.93 million

epoch: 0, time(min): 0.09, lr: 0.0001000, mean_loss: 1.4470
epoch: 1, time(min): 0.17, lr: 0.0001000, mean_loss: 1.1153
epoch: 2, time(min): 0.25, lr: 0.0001000, mean_loss: 0.8927
epoch: 3, time(min): 0.34, lr: 0.0001000, mean_loss: 0.7999
epoch: 4, time(min): 0.43, lr: 0.0001000, mean_loss: 0.6651
epoch: 5, time(min): 0.51, lr: 0.0001000, mean_loss: 1.0330
epoch: 6, time(min): 0.60, lr: 0.0000850, mean_loss: 0.9985
epoch: 7, time(min): 0.69, lr: 0.0000850, mean_loss: 1.0182
epoch: 8, time(min): 0.77, lr: 0.0000723, mean_loss: 0.9215
epoch: 9, time(min): 0.87, lr: 0.0000723, mean_loss: 0.8293
epoch: 10, time(min): 0.96, lr: 0.0000723, mean_loss: 0.5287
epoch: 11, time(min): 1.04, lr: 0.0000723, mean_loss: 0.6047
epoch: 12, time(min): 1.12, lr: 0.0000614, mean_loss: 0.5802
epoch: 13, time(min): 1.21, lr: 0.0000614, mean_loss: 0.4482
epoch: 14, time(min): 1.30, lr: 0.0000614, mean_loss: 0.3285
epoch: 15, time(min): 1.39, lr: 0.0000614, mean_loss

In [14]:
# # Loading pre-trained net
# net = attention_net(hidden_size, num_heads, num_blocks, seq_length).to(device)
# num_param = number_param(net)
# print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )
# checkpoint_file = "checkpoint/seq2seq_checkpoint_26-02-18--14-34-00.pkl"
# print(f'Loading pre-trained model {checkpoint_file}')
# if not os.path.exists(checkpoint_file):
#     print(f'Downloading {checkpoint_file} ...')
#     os.makedirs('checkpoint', exist_ok=True)
#     dropbox_url = "https://www.dropbox.com/scl/fi/8qcs50gy1tyxwif5zlb8p/seq2seq_checkpoint_26-02-18-14-34-00.pkl?rlkey=e01bo67muietydlhb3ybj5z0k&dl=1"
#     !wget -q "{dropbox_url}" -O "{checkpoint_file}" 
# checkpoint = torch.load(checkpoint_file, map_location=device)
# net.load_state_dict(checkpoint['net']) 


## Apply the network to unseen data


In [15]:
# Auto-regressive generation of the solution from a new/unseen data
seq_enc_in = torch.stack([torch.randperm(seq_length)[:10] for _ in range(batch_size)]).to(device)
seq_target = seq_enc_in.sort(dim=1)[0]
seq_dec_in = start_tokens # start tokens
batch_len = seq_enc_in.size(1)
for idx in range(batch_len):
    scores = net( seq_enc_in, seq_dec_in, pos_enc ) # size=(bs, seq_length+1, vocab_size), e.g. [4, 16, 17]
    scores_last_token = scores[:,-1,:].squeeze(dim=1) # extract the last token scores, size=[bs, num_tokens]
    probs_last_token = torch.softmax(scores_last_token, dim=1) # size=[1, num_tokens]
    idx_next_token = torch.max(probs_last_token, dim=1).indices.view(bs,1) # sample the next token, size=[bs,1]
    seq_dec_in = torch.cat((seq_dec_in, idx_next_token), dim=1) # append sampled token, size=[bs, num_tokens+1]

print('seq_enc_in:', seq_enc_in.size(), '\n', seq_enc_in, '\n')
print('ground truth:', seq_target.size(), '\n', seq_target, '\n')
print('prediction:', seq_dec_in[:,1:].size(), '\n', seq_dec_in[:,1:], '\n')
acc = (seq_dec_in[:,1:] == seq_target).sum().item() / seq_target.numel()
print('accuracy:', acc*100)
            

seq_enc_in: torch.Size([4, 10]) 
 tensor([[ 1,  5, 11,  2,  9, 14,  0,  6,  3,  8],
        [10,  5,  6, 12,  1,  7,  0,  2, 14,  4],
        [12,  0,  6,  5,  7, 13,  3,  1,  4,  2],
        [ 9,  1,  6, 12,  7,  3, 11,  0, 13,  4]], device='cuda:0') 

ground truth: torch.Size([4, 10]) 
 tensor([[ 0,  1,  2,  3,  5,  6,  8,  9, 11, 14],
        [ 0,  1,  2,  4,  5,  6,  7, 10, 12, 14],
        [ 0,  1,  2,  3,  4,  5,  6,  7, 12, 13],
        [ 0,  1,  3,  4,  6,  7,  9, 11, 12, 13]], device='cuda:0') 

prediction: torch.Size([4, 10]) 
 tensor([[ 0,  1,  2,  3,  5,  6,  8,  9, 11, 14],
        [ 0,  1,  2,  4,  5,  6,  7, 10, 12, 14],
        [ 0,  1,  2,  3,  4,  5,  6,  7, 12, 13],
        [ 0,  1,  3,  4,  6,  7,  9, 11, 12, 13]], device='cuda:0') 

accuracy: 100.0
